# 02 — Data preparation

**Machine Learning · M.EIC001 · Group G51**

This notebook covers the **data preparation** phase of CRISP-DM. It builds one table with one row per team-season, holding the targets of that season and features computed only from earlier seasons.

**Main rule:** we predict the day before the first game of season `t`, so features can only use data up to season `t-1`.

## 0. Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

pd.set_option("display.precision", 3)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

RANDOM_STATE = 42
DATA_DIR = Path("hockey_development_data") / "hockey_development_data"
FIRST_TARGET_YEAR = 1974  # first season with confID

## 1. Load and scope

We keep the NHL only. Earlier NHL seasons stay in the tables for now because they can feed history features, but only seasons from 1974 become target rows.

Free-text columns are dropped. `Scoring` rows without `GP` are also dropped: in 238 of the 249 NHL cases the player has no other row with stats in that year.

In [2]:
def load_nhl(name, drop=("note", "notes", "nameNote")):
    df = pd.read_csv(DATA_DIR / f"{name}.csv", low_memory=False)
    df = df.query("lgID == 'NHL'").copy()
    return df.drop(columns=list(drop), errors="ignore")

teams = load_nhl("Teams")
coaches = load_nhl("Coaches", drop=())  # notes marks co-coaches
scoring = load_nhl("Scoring").dropna(subset=["GP"])
goalies = load_nhl("Goalies").dropna(subset=["GP"])
master = pd.read_csv(DATA_DIR / "Master.csv", low_memory=False)

pd.Series({"Teams": len(teams), "Coaches": len(coaches), "Scoring": len(scoring),
           "Goalies": len(goalies), "Master": len(master)}, name="rows")

Teams       1265
Coaches     1522
Scoring    39852
Goalies     3660
Master      7469
Name: rows, dtype: int64

## 2. Targets

**Task (a).** `Teams.rank` is the **division** rank, so the conference rank is computed here. `pts_pct` is `Pts / (2 * G)`: `Pts = 2W + T + OTL` holds in every NHL row, and since 2005 shootout losses are already inside `OTL`. Teams level on points are separated by wins.

**Task (b).** A team changes coach if the head coach who ends the season is not the one who started it (first vs last `stint`). A rule on the number of coaches or on a minimum of games per spell does not work because short spells at the start are real early firings (e.g. CHI 2008, MTL 1995), interim spells between two different coaches do not change the result, and the real false positives follow an A → B → A pattern (OTT 2001 and NYI 2007, where an `interim` coached 1 or 2 games as a tribute). Rows marked `co-coach` in `notes` are dropped first (DET 1998, temporary co-coaches while the head coach was out). The `interim` note is only filled in 4 rows, so it is not used.

In [3]:
teams["pts_pct"] = teams["Pts"] / (2 * teams["G"])

ranked = (teams[teams["confID"].notna()]
          .sort_values(["year", "confID", "pts_pct", "W"], ascending=[True, True, False, False]))
teams["conf_rank"] = ranked.groupby(["year", "confID"]).cumcount() + 1

head_coaches = (coaches[~coaches["notes"].fillna("").str.contains("co-coach")]
                .sort_values(["year", "tmID", "stint"]))
coach_changed = (head_coaches.groupby(["year", "tmID"])["coachID"]
                 .agg(lambda s: int(s.iloc[0] != s.iloc[-1]))
                 .rename("coach_changed").reset_index())
teams = teams.merge(coach_changed, on=["year", "tmID"], how="left")
teams["coach_changed"] = teams["coach_changed"].fillna(0).astype(int)

targets_1974 = teams[teams["year"] >= FIRST_TARGET_YEAR]
pd.Series({"team-seasons": len(targets_1974),
           "coach change rate": targets_1974["coach_changed"].mean(),
           "rows without conf_rank": targets_1974["conf_rank"].isna().sum()}, name="targets")

team-seasons              842.000
coach change rate           0.194
rows without conf_rank      0.000
Name: targets, dtype: float64

## 3. Team features

These are computed per season first and only moved to the next season in section 4.

Each one is converted to a z-score inside its season, so that values are relative to the league of that year and comparable across eras.

In [4]:
teams["goal_diff_g"] = (teams["GF"] - teams["GA"]) / teams["G"]
teams["pim_g"] = teams["PIM"] / teams["G"]
teams["pp_pct"] = teams["PPG"] / teams["PPC"]
teams["pk_pct"] = 1 - teams["PKG"] / teams["PKC"]

SEASON_STATS = ["pts_pct", "goal_diff_g", "pim_g", "pp_pct", "pk_pct"]

def season_zscore(s):
    return (s - s.mean()) / s.std()

for col in SEASON_STATS:
    teams[f"{col}_z"] = teams.groupby("year")[col].transform(season_zscore)

teams[[f"{c}_z" for c in SEASON_STATS]].describe().loc[["count", "mean", "std"]]

,pts_pct_z,goal_diff_g_z,pim_g_z,pp_pct_z,pk_pct_z
count,1.265e+03,1.265e+03,1.265e+03,9.620e+02,9.620e+02
mean,3.704e-17,7.021e-19,-2.826e-17,-2.954e-17,3.370e-16
std,9.633e-01,9.633e-01,9.633e-01,9.763e-01,9.763e-01


## 4. Franchise history

`tmID` changes when a team moves or is renamed, so history follows `franchID`. `shift` takes the previous season the franchise played, so the season before 2005 is 2003 (2004 was the lockout). `year - 1` would get this wrong.

- `_lag1`: value in the previous season
- `_avg3`: mean of the previous three seasons, shifted so season `t` is never included

Teams filtered out of the NHL have no earlier row here, so the four franchises that joined from the WHA in 1979 are treated as new teams.

In [5]:
teams = teams.sort_values(["franchID", "year"]).reset_index(drop=True)
by_franchise = teams.groupby("franchID")

history = {}
for col in [f"{c}_z" for c in SEASON_STATS]:
    history[f"{col}_lag1"] = by_franchise[col].shift(1)
    history[f"{col}_avg3"] = by_franchise[col].transform(
        lambda s: s.shift(1).rolling(3, min_periods=1).mean())
history["coach_changed_lag1"] = by_franchise["coach_changed"].shift(1)

teams = pd.concat([teams, pd.DataFrame(history)], axis=1)
teams["is_new_team"] = teams["pts_pct_z_lag1"].isna().astype(int)

TEAM_FEATURES = list(history) + ["is_new_team"]
print(len(TEAM_FEATURES), "team features")

12 team features


## 5. Player features

These features describe the players of each franchise, using only data from before season `t`.

- **Roster of `t`:** the skaters who **ended season `t-1` with that franchise** (their last `stint` of the season). Season `t` is not used at all, so there is no leakage.
- **Stats:** each player's stats in `t-1`, summed over every team he played for that season. The previous season is the previous one the league played (2005 → 2003, as in section 4).
- Goalies are left out here and get their own features.

| Feature | Meaning |
|---|---|
| `roster_pts_top12` | points in `t-1` of the 12 best scorers of the roster |
| `roster_pm_top12` | plus/minus in `t-1` of those 12 |
| `roster_age_core` | mean age in season `t` of the 15 players with most games in `t-1` |
| `roster_n_award_3y` | players with an All-Star team or a major award in the last 3 seasons |

A top-6 version of the points was also built but it has a 0.93 correlation with the top-12, so only the top-12 is kept. Like the team stats, every feature is a z-score inside its season. New franchises have no roster in `t-1`, so their values stay missing, like the other history features.

**Why not the roster of season `t`?** The data has no opening-day roster. Using the players with `stint == 1` in `t` would also include players who only joined during the season, which is not known the day before the first game. That version was tested and gave the same result (mean Spearman 0.538 against 0.535 for this one, with the linear check of section 8), so the version without leakage is kept. The cost is that moves made in the summer (trades, signings, retirements) are not captured.

In [6]:
awards = load_nhl("AwardsPlayers")
QUALITY_AWARDS = ["First Team All-Star", "Second Team All-Star", "All-Rookie", "Hart", "Art Ross",
                  "Norris", "Selke", "Calder", "Pearson", "Richard", "Conn Smythe", "Lady Byng"]

seasons = np.sort(teams["year"].unique())
next_season = dict(zip(seasons[:-1], seasons[1:]))
last3_seasons = {y: seasons[seasons < y][-3:] for y in seasons}

skaters = (scoring[scoring["pos"] != "G"]
           .merge(teams[["year", "tmID", "franchID"]], on=["year", "tmID"]))

# stats of each player in a season, summed over all his teams
player_season = (skaters.groupby(["playerID", "year"])[["GP", "Pts", "+/-"]].sum()
                 .rename(columns={"GP": "gp_prev", "Pts": "pts_prev", "+/-": "pm_prev"})
                 .reset_index())

# roster of t = players whose last stint of t-1 was with the franchise
roster = (skaters.sort_values("stint").groupby(["playerID", "year"]).tail(1)
          [["playerID", "year", "franchID"]]
          .merge(player_season, on=["playerID", "year"])
          .rename(columns={"year": "prev_year"}))
roster["year"] = roster["prev_year"].map(next_season)
roster = roster.dropna(subset=["year"]).astype({"year": int})
roster = roster.merge(master[["playerID", "birthYear"]], on="playerID", how="left")
roster["age"] = roster["year"] - roster["birthYear"]

# 1 if the player won a quality award in one of the last 3 league seasons
award_years = awards.loc[awards["award"].isin(QUALITY_AWARDS), ["playerID", "year"]].drop_duplicates()
recent = (roster[["playerID", "year"]].drop_duplicates()
          .assign(award_year=lambda d: d["year"].map(last3_seasons)).explode("award_year")
          .merge(award_years.rename(columns={"year": "award_year"}).assign(award=1),
                 on=["playerID", "award_year"], how="left")
          .groupby(["playerID", "year"])["award"].max().fillna(0).rename("award_3y").reset_index())
roster = roster.merge(recent, on=["playerID", "year"], how="left")

def roster_features(g):
    top12 = g.nlargest(12, "pts_prev")
    core = g.nlargest(15, "gp_prev")
    return pd.Series({
        "roster_pts_top12": top12["pts_prev"].sum(),
        "roster_pm_top12": top12["pm_prev"].sum(),
        "roster_age_core": core["age"].mean(),
        "roster_n_award_3y": g["award_3y"].sum(),
    })

roster_feats = roster.groupby(["year", "franchID"]).apply(roster_features, include_groups=False).reset_index()
ROSTER_RAW = [c for c in roster_feats.columns if c.startswith("roster_")]

teams = teams.merge(roster_feats, on=["year", "franchID"], how="left")
for col in ROSTER_RAW:
    teams[f"{col}_z"] = teams.groupby("year")[col].transform(season_zscore)
ROSTER_FEATURES = [f"{c}_z" for c in ROSTER_RAW]

# leakage checks: no stats from season t, and Detroit 2005 uses 2003
assert (roster["prev_year"] < roster["year"]).all()
assert (roster.query("year == 2005 and franchID == 'DET'")["prev_year"] == 2003).all()

# how each feature relates to pts_pct, next to the previous-season baseline
recent_teams = teams[teams["year"] >= FIRST_TARGET_YEAR]
pd.Series({c: spearmanr(recent_teams[c], recent_teams["pts_pct"], nan_policy="omit")[0]
           for c in ["pts_pct_z_lag1"] + ROSTER_FEATURES}, name="spearman with pts_pct").round(3)

pts_pct_z_lag1         0.595
roster_pts_top12_z     0.461
roster_pm_top12_z      0.567
roster_age_core_z      0.113
roster_n_award_3y_z    0.418
Name: spearman with pts_pct, dtype: float64

`roster_pm_top12` is the strongest, but it overlaps a lot with `pts_pct_z_lag1` (correlation 0.90), because plus/minus mostly measures how the team did. Points (0.78) and awards (0.54) overlap less. Whether they add anything to the team features is checked in section 8.

## 6. Goalie features

The starting goalie has a big effect on how many goals a team allows, so he gets his own features. The same rule as section 5 applies: the goalies of season `t` are the ones who **ended `t-1` with the franchise**, and the **starter** is the one of them with most minutes in `t-1`.

| Feature | Meaning |
|---|---|
| `goalie_gaa_3y` | goals against per 60 minutes of the starter, over his last 3 league seasons |
| `goalie_sv_3y` | save percentage of the starter (`1 - GA / SA`), over the same 3 seasons |
| `goalie_starter_share` | share of the franchise's goalie minutes in `t-1` played by the starter |
| `goalie_starter_age` | age of the starter in season `t` |

- **Why 3 seasons:** save percentage changes a lot from one season to the next, so the 3 seasons are pooled (totals of `GA`, `SA` and `Min`), which also gives more weight to the seasons where he played more.
- **GAA vs save %:** GAA also depends on how many shots the defence allows. Save percentage isolates the goalie better, but it needs `SA`.
- **Missing `SA`:** `SA` only exists from 1982, so `goalie_sv_3y` is missing when any season of the window is before 1982 (target seasons up to 1984, about a quarter of the panel). It is **not imputed here**: the imputation has to be fitted inside each training fold, so it is done in the modelling `Pipeline` with a model-based imputer (e.g. `IterativeImputer` from GAA and the team stats).

In [7]:
gk = goalies.merge(teams[["year", "tmID", "franchID"]], on=["year", "tmID"])

# stats of each goalie in a season, summed over all his teams (SA stays missing before 1982)
goalie_season = gk.groupby(["playerID", "year"])[["Min", "GA", "SA"]].sum(min_count=1).reset_index()

# goalies of t = those whose last stint of t-1 was with the franchise
gk_roster = (gk.sort_values("stint").groupby(["playerID", "year"]).tail(1)[["playerID", "year", "franchID"]]
             .merge(goalie_season, on=["playerID", "year"])
             .rename(columns={"year": "prev_year"}))
gk_roster["year"] = gk_roster["prev_year"].map(next_season)
gk_roster = gk_roster.dropna(subset=["year"]).astype({"year": int})
gk_roster["team_min"] = gk_roster.groupby(["year", "franchID"])["Min"].transform("sum")

starters = (gk_roster.sort_values("Min", ascending=False)
            .groupby(["year", "franchID"]).head(1)
            .merge(master[["playerID", "birthYear"]], on="playerID", how="left"))
starters["goalie_starter_share"] = starters["Min"] / starters["team_min"]
starters["goalie_starter_age"] = starters["year"] - starters["birthYear"]

# pooled stats of the starter over his last 3 league seasons (only the seasons he played)
window = (starters[["playerID", "year", "franchID"]]
          .assign(season=lambda d: d["year"].map(last3_seasons)).explode("season")
          .merge(goalie_season.rename(columns={"year": "season"}), on=["playerID", "season"]))
pooled = window.groupby(["year", "franchID"]).agg(
    min_3y=("Min", "sum"), ga_3y=("GA", "sum"), sa_3y=("SA", "sum"),
    sa_complete=("SA", lambda s: s.notna().all()))
pooled["goalie_gaa_3y"] = pooled["ga_3y"] * 60 / pooled["min_3y"]
pooled["goalie_sv_3y"] = (1 - pooled["ga_3y"] / pooled["sa_3y"]).where(pooled["sa_complete"])

goalie_feats = (starters.set_index(["year", "franchID"])[["goalie_starter_share", "goalie_starter_age"]]
                .join(pooled[["goalie_gaa_3y", "goalie_sv_3y"]]).reset_index())
GOALIE_RAW = ["goalie_gaa_3y", "goalie_sv_3y", "goalie_starter_share", "goalie_starter_age"]

teams = teams.merge(goalie_feats, on=["year", "franchID"], how="left")
for col in GOALIE_RAW:
    teams[f"{col}_z"] = teams.groupby("year")[col].transform(season_zscore)
GOALIE_FEATURES = [f"{c}_z" for c in GOALIE_RAW]

# leakage check: no goalie stats from season t
assert (gk_roster["prev_year"] < gk_roster["year"]).all()
assert (window["season"] < window["year"]).all()

recent_teams = teams[teams["year"] >= FIRST_TARGET_YEAR]
pd.DataFrame({
    "spearman with pts_pct": {c: spearmanr(recent_teams[c], recent_teams["pts_pct"], nan_policy="omit")[0]
                              for c in GOALIE_FEATURES},
    "missing share": recent_teams[GOALIE_FEATURES].isna().mean(),
}).round(3)

,spearman with pts_pct,missing share
goalie_gaa_3y_z,-0.434,0.018
goalie_sv_3y_z,0.219,0.262
goalie_starter_share_z,0.073,0.018
goalie_starter_age_z,0.066,0.018


`goalie_gaa_3y` is the most related to the points of the next season (negative, as expected: fewer goals against, more points). `goalie_sv_3y` is weaker, partly because it only exists from 1985 on. Share and age say little by themselves, but they may matter in interaction with the other features (e.g. an old starter with a weak backup).

## 7. Build the panel

One row per team-season from 1974. It keeps the keys, the targets and the features. Raw columns of season `t` are left out, which is what prevents leakage.

Missing values stay as they are: imputation and scaling must be fitted inside each training fold, in a `Pipeline`, during modelling.

In [8]:
KEYS = ["year", "franchID", "tmID", "confID", "divID"]
TARGETS = ["pts_pct", "conf_rank", "coach_changed"]
FEATURES = TEAM_FEATURES + ROSTER_FEATURES + GOALIE_FEATURES

panel = (teams.loc[teams["year"] >= FIRST_TARGET_YEAR, KEYS + TARGETS + FEATURES]
              .sort_values(["year", "confID", "conf_rank"])
              .reset_index(drop=True))

# checks on the table
assert not panel.duplicated(["year", "tmID"]).any()
assert panel["conf_rank"].notna().all()
assert all(c.endswith(("_lag1", "_avg3")) or c == "is_new_team" for c in TEAM_FEATURES)

# leakage check: a lagged feature must match the franchise's previous season
det = teams.query("franchID == 'DET'").set_index("year")
assert np.isclose(det.loc[2005, "pts_pct_z_lag1"], det.loc[2003, "pts_pct_z"])

print(panel.shape, "| seasons:", panel["year"].nunique(), "| new teams:", panel["is_new_team"].sum())
panel.head()

(842, 28) | seasons: 35 | new teams: 15


,year,franchID,tmID,confID,divID,pts_pct,conf_rank,coach_changed,pts_pct_z_lag1,pts_pct_z_avg3,goal_diff_g_z_lag1,goal_diff_g_z_avg3,pim_g_z_lag1,pim_g_z_avg3,pp_pct_z_lag1,pp_pct_z_avg3,pk_pct_z_lag1,pk_pct_z_avg3,coach_changed_lag1,is_new_team,roster_pts_top12_z,roster_pm_top12_z,roster_age_core_z,roster_n_award_3y_z,goalie_gaa_3y_z,goalie_sv_3y_z,goalie_starter_share_z,goalie_starter_age_z
0,1974,PHI,PHI,CC,PT,0.706,1.0,0,1.553,0.454,1.430,0.489,3.216,2.781,0.627,0.900,2.110,0.999,0.0,0,0.390,1.191,-0.327,0.395,-1.816,NaN,1.883,0.085
1,1974,NYR,NYR,CC,PT,0.550,2.0,0,0.731,1.018,0.643,1.083,-0.687,-0.354,2.158,1.281,0.148,0.745,1.0,0,1.359,0.266,2.008,2.119,-0.253,NaN,0.457,1.714
2,1974,NYI,NYI,CC,PT,0.550,3.0,0,-1.005,-1.547,-0.853,-1.489,0.495,0.112,-1.104,-1.384,-1.271,-1.439,0.0,0,-1.503,-0.540,-1.451,-0.180,1.218,NaN,-0.473,-1.273
3,1974,VAN,VAN,CC,SM,0.537,4.0,0,-0.868,-1.064,-0.945,-1.124,-0.001,0.295,0.092,-0.199,-1.808,-1.431,1.0,0,-0.904,-0.793,-1.927,-0.754,0.381,NaN,0.937,0.356
4,1974,STL,STL,CC,SM,0.525,5.0,1,-0.639,-0.393,-0.551,-0.416,0.785,1.027,-1.049,-1.037,0.403,-0.264,1.0,0,-0.746,-0.784,0.711,-0.754,0.190,NaN,-0.827,0.085


## 8. Baseline

Each new group of features should be compared with this. The baseline orders every conference by the previous season, and new teams get the mean. Each conference-season gets its own Spearman correlation, then these are averaged.

Predicted ranks use `method="average"`. About 20% of the teams tie with another team of their conference on the previous season, and `method="first"` would break those ties by row order. The panel is sorted by the true `conf_rank`, so that leaked the answer and made the baseline look better (0.549 instead of 0.536 for 1995-2009).

In [9]:
def rank_scores(df, score_col):
    pred = df.groupby(["year", "confID"])[score_col].rank(ascending=False, method="average")
    spearman = [spearmanr(pred[g.index], g["conf_rank"])[0] for _, g in df.groupby(["year", "confID"])]
    return {"spearman": np.mean(spearman), "rank_mae": (pred - df["conf_rank"]).abs().mean()}

baseline = panel.assign(score=panel["pts_pct_z_lag1"].fillna(0))  # z-score mean is 0

pd.DataFrame({
    "1995-2009": rank_scores(baseline[baseline["year"] >= 1995], "score"),
    "2005-2009": rank_scores(baseline[baseline["year"] >= 2005], "score"),
}).T

,spearman,rank_mae
1995-2009,0.536,3.072
2005-2009,0.484,3.373


**Quick check of the feature groups.** A linear regression on `pts_pct`, trained on the seasons before 1995 and tested on 1995-2009, with the same metric. Missing values are set to 0 (the season mean of a z-score). This is only a sanity check with one split, so differences of about 0.01 are noise: the real comparison is done in modelling, with cross-validation over seasons.

In [10]:
from sklearn.linear_model import LinearRegression

def linear_check(cols, first_test_year=1995):
    data = panel.copy()
    data[cols] = data[cols].fillna(0)
    train, test = data[data["year"] < first_test_year], data[data["year"] >= first_test_year]
    model = LinearRegression().fit(train[cols], train["pts_pct"])
    return rank_scores(test.assign(pred=model.predict(test[cols])), "pred")

groups = {
    "previous pts_pct only": ["pts_pct_z_lag1"],
    "team": TEAM_FEATURES,
    "team + roster": TEAM_FEATURES + ROSTER_FEATURES,
    "team + goalie": TEAM_FEATURES + GOALIE_FEATURES,
    "team + roster + goalie": FEATURES,
}
pd.DataFrame({name: linear_check(cols) for name, cols in groups.items()}).T

,spearman,rank_mae
previous pts_pct only,0.536,3.072
team,0.556,3.032
team + roster,0.544,3.032
team + goalie,0.548,3.047
team + roster + goalie,0.547,3.032


The team features beat the previous season alone (0.556 against 0.536). Adding the roster or the goalie features does not help a linear model (0.544 to 0.548): most of what they say about a team is already in its results, and `goalie_sv_3y` is 0 for every training season before 1985, so its coefficient is learned on a very different distribution from the test set. They are kept for modelling, where proper imputation, non-linear models and feature selection can still make use of them.

## 9. Next steps

- [ ] Decide the rank tie-break after talking to the professor
- [x] Roster features from `Scoring`, `Master` and `AwardsPlayers` (section 5)
- [x] Goalie features from `Goalies` (section 6)
- [ ] Model-based imputation of `goalie_sv_3y` inside the training folds (`IterativeImputer`)
- [ ] Coach features from `Coaches`: seasons in the job, career win percentage
- [ ] Form features from `TeamSplits` and `TeamsPost`: home and road records, late-season results, playoff round reached
- [ ] Outlier handling for linear models (`RobustScaler` or winsorizing)
- [ ] Feature selection and SMOTE inside the training folds

**Notes for the report**

- Roster features (section 5): the roster of `t` is the one that ended `t-1`, so summer moves are not captured. Using `stint == 1` of `t` would leak players who joined during the season, and it gave the same result in a quick test (0.538 vs 0.535), so it was dropped.
- Baseline: ranks must use `method="average"`. With `"first"`, ties (20% of teams) were broken by the true rank, which inflated the baseline from 0.536 to 0.549.
- Roster and goalie features did not improve a linear model over the team features in a quick check (section 8). To revisit in modelling.
- `goalie_sv_3y` is missing up to 1984 because `SA` only starts in 1982.